# PhysIR — Stage 2: Combined Physics Losses (Kaggle)

**Purpose:** All three physics losses active simultaneously from epoch 1.
Verify they combine without interference and find a PSNR ceiling above best solo pilot (C: 24.94 dB).

**Pre-normalized (tempered) weights** — derived from pilot ep50 loss contributions:

| Loss | Solo @ep50 (% of base) | Stage 2 weight | Target |
|------|------------------------|----------------|--------|
| retinex | 28% (w=0.10) | **0.18** | ~0.5× base |
| blur    | 13% (w=0.05) | **0.17** | ~0.5× base |
| phase   | **383%** (w=0.05) | **0.02** | ~1.5× base (stays lead) |

Phase was both dominant *and* the best performer (+0.70 dB), so we keep it as the lead physics
loss but pull it back from ~4× base so retinex/blur aren't drowned. Stage 3 sweep refines from here.

**Phase gating:** `phase4_start: 1` → phase=4 from epoch 1, all three gates satisfied.

**Generalization:** LOLv2 test split evaluated every 10 epochs (separate log, never affects best-checkpoint selection).

**Reference targets:**
| Run | PSNR | SSIM | LPIPS |
|-----|------|------|-------|
| A baseline | 24.2437 | 0.7955 | 0.3493 |
| B retinex  | 24.6115 | 0.8067 | 0.3453 |
| C freq     | 24.9414 | 0.8393 | 0.2733 |
| D blur     | 24.0791 | 0.7963 | 0.3478 |

**Watch:** loss balance monitor — if any loss >60%, lower its weight. PSNR should climb past 24.94 by ep 25–30 if losses cooperate.

**Accelerator:** Set *Settings → Accelerator → GPU T4 x2*.

## 1. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths — edit before running

In [ ]:
import os, sys

# ── EDIT THESE ────────────────────────────────────────────────────────────────
CODE_DIR     = '/kaggle/working/PhysIR'
LOLBLUR_ROOT = '/kaggle/input/lolblur/LOLBlur'
# LOLv2 generalization root — a PARENT dir that contains LOL-v2/ laid out as:
#   <LOLV2_ROOT>/LOL-v2/Real_captured/test/{Low,Normal}
#   <LOLV2_ROOT>/LOL-v2/Synthetic/test/{Low,Normal}
# The All_LOL reader skips missing sub-sets gracefully. Set to None to disable gen eval.
LOLV2_ROOT   = '/kaggle/input/lolv2'   # EDIT or set None to skip
CHECKPOINT_DIR = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Code:        {CODE_DIR}')
print(f'Dataset:     {LOLBLUR_ROOT}')
print(f'LOLv2 (gen): {LOLV2_ROOT}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## 3. Clone PhysIR repo *(skip if mounted as Kaggle dataset)*

In [ ]:
if not os.path.isdir(CODE_DIR):
    GITHUB_URL = 'https://github.com/<your-username>/PhysIR.git'  # EDIT
    !git clone {GITHUB_URL} {CODE_DIR}
else:
    print('Code directory already present.')

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('cwd:', os.getcwd())

## 4. Train/val split

In [ ]:
import random
from pathlib import Path

SUBS       = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC  = Path(LOLBLUR_ROOT) / 'train'
SPLIT_ROOT = Path(WORKING_DIR) / 'LOLBlur_split'

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips   = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'Clips — train: {len(train_clips)}, val: {len(val_clips)}')

for split_name, clip_list in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        dst_base = SPLIT_ROOT / split_name / sub
        dst_base.mkdir(parents=True, exist_ok=True)
        for clip in clip_list:
            src_path = (TRAIN_SRC / sub / clip).resolve()
            dst_path = dst_base / clip
            if not dst_path.exists():
                dst_path.symlink_to(src_path)

PHYSIR_TRAIN = str(SPLIT_ROOT / 'physir_train')
PHYSIR_VAL   = str(SPLIT_ROOT / 'physir_val')
print(f'  train: {PHYSIR_TRAIN}')
print(f'  val:   {PHYSIR_VAL}')

## 5. Config — Stage 2 combined

In [ ]:
import yaml

cfg = {
    'datasets': {
        'name': 'LOLBlur',
        'train': {
            'low_path':   f'{PHYSIR_TRAIN}/low_blur_noise',
            'high_path':  f'{PHYSIR_TRAIN}/high_sharp_scaled',
            'crop_size':  256, 'batch_size': 4, 'n_workers': 4, 'use_flips': True,
        },
        'val': {
            'test_path': PHYSIR_VAL, 'batch_size_test': 1,
            'verbose': False, 'n_workers': 4,
        },
    },
    'network': {
        'name': 'DarkIR', 'img_channels': 3, 'width': 32,
        'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
        'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
        'dilations': [1, 4, 9], 'extra_depth_wise': True,
    },
    'save': {
        'pretrained': None,
        'path': f'{CHECKPOINT_DIR}/stage2_combined_latest.pt',
        'best': f'{CHECKPOINT_DIR}/stage2_combined_best.pt',
        'save_freq': 5, 'eval_freq': 5, 'gen_eval_freq': 10,
        'perceptual_eval_epochs': [10, 20, 35],
    },
    'pixel_criterion': 'l1', 'perceptual': False,
    'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
    'frequency': False,
    'enhance': True, 'enhance_weight': 0.01,
    'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
    'physics_losses': {
        'enabled': True,
        'phase2_start': 9999,   # unused — phase4_start fires all gates
        'phase3_start': 9999,   # unused
        'phase4_start': 1,      # phase=4 from epoch 1; all three losses active
        # Pre-normalized (tempered) weights — see notebook header for derivation.
        'retinex': True,  'retinex_weight': 0.18,   # ~0.5x base
        'blur':    True,  'blur_weight':    0.17,   # ~0.5x base
        'phase':   True,  'phase_weight':   0.02,   # ~1.5x base (lead loss)
    },
    'train': {
        'epochs': 35, 'lr_initial': 0.00028, 'weight_decay': 0.001,
        'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0,
    },
    'wandb': {'init': False, 'name': 'stage2_combined'},
    'Resize': False,
}

# LOLv2 generalization block — added only if LOLV2_ROOT is set and exists.
# The All_LOL reader picks up <LOLV2_ROOT>/LOL-v2/Real_captured/test and Synthetic/test.
if LOLV2_ROOT and os.path.isdir(LOLV2_ROOT):
    cfg['datasets_gen'] = {
        'name': 'All_LOL',
        'train': {'n_workers': 2, 'verbose': True},
        'val': {'test_path': LOLV2_ROOT, 'batch_size_test': 1,
                'verbose': True, 'n_workers': 2},
    }
    gen_status = f'enabled → {LOLV2_ROOT}'
else:
    gen_status = 'disabled (LOLV2_ROOT not set or missing)'

cfg_path = f'{WORKING_DIR}/stage2_combined_kaggle.yml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

pl = cfg['physics_losses']
print(f'Config written: {cfg_path}')
print(f'  retinex: w={pl["retinex_weight"]}   blur: w={pl["blur_weight"]}   phase: w={pl["phase_weight"]}')
print(f'  phase4_start: {pl["phase4_start"]}  (all gates active from ep 1)')
print(f'  epochs: {cfg["train"]["epochs"]}')
print(f'  LOLv2 gen eval: {gen_status}')

## 6. Resume detection

In [ ]:
latest_ckpt = f'{CHECKPOINT_DIR}/stage2_combined_latest.pt'

if os.path.isfile(latest_ckpt):
    size_mb = os.path.getsize(latest_ckpt) / 1e6
    import torch as _t
    _ckpt = _t.load(latest_ckpt, map_location='cpu', weights_only=False)
    resume_ep = _ckpt.get('epoch', '?')
    del _ckpt
    print(f'✅ Checkpoint found ({size_mb:.1f} MB) — resuming from epoch {resume_ep}')
else:
    print('No checkpoint found — training from scratch.')

## 7. Train

In [ ]:
!torchrun --nproc_per_node=2 {CODE_DIR}/train.py -p {cfg_path}

## 8. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

metrics_dir = Path(CODE_DIR) / '.metrics' / 'stage2_combined'
log_path  = metrics_dir / 'training_log.csv'
perc_path = metrics_dir / 'perceptual_log.csv'
gen_path  = metrics_dir / 'generalization_log.csv'

REFERENCES = {
    'A baseline': (24.2437, 0.7955, 0.3493),
    'B retinex':  (24.6115, 0.8067, 0.3453),
    'C freq':     (24.9414, 0.8393, 0.2733),
    'D blur':     (24.0791, 0.7963, 0.3478),
}

if not log_path.exists():
    print('Log not found — training may still be running or path is wrong.')
else:
    df = pd.read_csv(log_path)
    df.replace('', float('nan'), inplace=True)
    for col in ['val_psnr', 'val_ssim', 'val_lpips']:
        if col not in df.columns:
            df[col] = float('nan')
    val = df.dropna(subset=['val_psnr'])

    fig, axes = plt.subplots(1, 6, figsize=(30, 4))
    ax = iter(axes)

    # Loss breakdown
    loss_cols = [c for c in df.columns if c.startswith('loss_')]
    df.plot(x='epoch', y=loss_cols, ax=next(ax), title='Loss components')

    # PSNR with reference lines
    ax_psnr = next(ax)
    val.plot(x='epoch', y='val_psnr', ax=ax_psnr, title='Val PSNR (dB)', marker='o', label='Stage2')
    for name, (psnr, _, _) in REFERENCES.items():
        ax_psnr.axhline(psnr, linestyle='--', alpha=0.5, label=name)
    ax_psnr.legend(fontsize=7)

    val.plot(x='epoch', y='val_ssim',  ax=next(ax), title='Val SSIM',    marker='o')
    val.plot(x='epoch', y='val_lpips', ax=next(ax), title='Val LPIPS ↓', marker='o')

    # Generalization panel — one line per OOD dataset (lolv2, lolv2_synth, ...)
    ax_gen = next(ax)
    gen = None
    if gen_path.exists():
        gen = pd.read_csv(gen_path)
        gen.replace('', float('nan'), inplace=True)
        for key, sub in gen.dropna(subset=['val_psnr']).groupby('dataset'):
            ax_gen.plot(sub['epoch'], sub['val_psnr'], marker='s', label=key)
        ax_gen.set_title('Generalization PSNR (dB)')
        ax_gen.legend(fontsize=7)
    else:
        ax_gen.set_title('Generalization: not evaluated')

    if perc_path.exists():
        perc = pd.read_csv(perc_path)
        ax_perc = next(ax)
        perc.plot(x='epoch', y=['nrqm', 'musiq'], ax=ax_perc,
                  title='NRQM↑ / MUSIQ↑  |  NIQE↓ (red)', marker='o')
        ax_perc.twinx().plot(perc['epoch'], perc['niqe'],
                             color='red', linestyle='--', marker='s', label='NIQE↓')
    else:
        next(ax).set_title('Perceptual: pending (ep 10/20/35)')

    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/stage2_combined_curves.png', dpi=120)
    plt.show()

    print('\n=== Stage 2 Summary ===')
    if len(val):
        best_psnr = val['val_psnr'].max()
        best_ep   = val.loc[val['val_psnr'].idxmax(), 'epoch']
        print(f'  Best PSNR : {best_psnr:.4f} dB  @ epoch {int(best_ep)}')
        print(f'  vs C freq : {best_psnr - 24.9414:+.4f} dB')
        print(f'  vs A base : {best_psnr - 24.2437:+.4f} dB')
        if val['val_ssim'].notna().any():
            print(f'  Best SSIM : {val["val_ssim"].max():.4f}')
        if val['val_lpips'].notna().any():
            print(f'  Best LPIPS: {val["val_lpips"].min():.4f}')
        print()
        if best_psnr > 24.9414:
            print('  ✅ Exceeds best solo pilot (C) — losses cooperate. Proceed to weight sweep.')
        elif best_psnr > 24.2437:
            print('  ⚠️  Above baseline but below C solo. Possible interference. Check loss balance.')
        else:
            print('  ❌ Below baseline — loss interference. Lower weights before sweep.')

    if gen is not None and len(gen.dropna(subset=['val_psnr'])):
        print('\n  Generalization (OOD), per dataset:')
        for key, sub in gen.dropna(subset=['val_psnr']).groupby('dataset'):
            print(f'    {key:14s} ep{int(sub["epoch"].iloc[0])}→ep{int(sub["epoch"].iloc[-1])}: '
                  f'{sub["val_psnr"].iloc[0]:.2f} → {sub["val_psnr"].iloc[-1]:.2f} dB '
                  f'(best {sub["val_psnr"].max():.2f})')

    if perc_path.exists():
        last = perc.iloc[-1]
        ep   = int(last['epoch'])
        print(f'\n  NIQE  ep{ep}: {last["niqe"]:.4f}  (lower=better)')
        print(f'  NRQM  ep{ep}: {last["nrqm"]:.4f}  (higher=better)')
        print(f'  MUSIQ ep{ep}: {last["musiq"]:.4f}  (higher=better)')